# MoE-LoRA Runtime Study Analysis

This notebook reads the artifacts from `scripts/local/moe_lora_runtime_study.py` and summarizes runtime, memory, and realized sparsity for `LoRA`, `HMoRA`, `AdaMoLE`, and `MoELoRA`.

Expected inputs:

- `results.jsonl`
- `results.csv`


In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from statistics import mean, median

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

RESULT_PATH = Path('/tmp/moe_lora_runtime_study/results.jsonl')
rows = [json.loads(line) for line in RESULT_PATH.read_text().splitlines() if line.strip()]
len(rows)


In [ ]:
def grouped_summary(records, group_keys):
    grouped = defaultdict(list)
    for row in records:
        key = tuple(row[key] for key in group_keys)
        grouped[key].append(row)
    result = []
    for key, values in sorted(grouped.items()):
        result.append({
            **dict(zip(group_keys, key)),
            'runs': len(values),
            'mean_tps': mean(v['tokens_per_second'] for v in values),
            'median_tps': median(v['tokens_per_second'] for v in values),
            'mean_peak_memory_bytes': mean((v['peak_memory_bytes'] or 0) for v in values),
            'mean_logical_active_experts': mean(v['logical_active_experts'] for v in values),
            'mean_executed_expert_fraction': mean(v['executed_expert_fraction'] for v in values),
        })
    return result

summary = grouped_summary(rows, ['method', 'routing_mode', 'pass_mode'])
summary


In [ ]:
controlled = [row for row in rows if row['routing_mode'] == 'controlled_routing' and row['pass_mode'] == 'forward_backward']
grouped_summary(controlled, ['method', 'controlled_active_experts'])


In [ ]:
native = [row for row in rows if row['routing_mode'] == 'native_routing' and row['pass_mode'] == 'forward_backward']
grouped_summary(native, ['method'])


In [ ]:
if plt is not None:
    plot_rows = [row for row in controlled if row['method'] in {'hmora', 'adamole', 'moelora'}]
    methods = ['hmora', 'adamole', 'moelora']
    fig, ax = plt.subplots(figsize=(8, 5))
    for method in methods:
        subset = sorted(
            [row for row in plot_rows if row['method'] == method],
            key=lambda row: (row['controlled_active_experts'], row['num_experts'], row['hidden_size'], row['seq_len']),
        )
        xs = [row['logical_active_experts'] for row in subset]
        ys = [row['tokens_per_second'] for row in subset]
        ax.scatter(xs, ys, label=method, alpha=0.75)
    ax.set_xlabel('Logical active experts per position')
    ax.set_ylabel('Tokens / second')
    ax.set_title('Controlled-routing throughput vs effective sparsity')
    ax.legend()
    plt.show()
else:
    print('matplotlib is not available in this environment')
